## Run Below to Connect via Google Colab

In [1]:
!pip install -q "cloud-sql-python-connector[pg8000]" sqlalchemy pandas

from google.colab import auth, userdata
auth.authenticate_user()   # must be your @g.harvard.edu account

DB_PASSWORD = userdata.get('DB_PASSWORD')

import sqlalchemy
from google.cloud.sql.connector import Connector
connector = Connector()

def _getconn():
    return connector.connect(
        "harvard-capstone-499102:us-central1:h5n1-pg", "pg8000",
        user="h5n1_app", password=DB_PASSWORD, db="h5n1",
    )

engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=_getconn)

import pandas as pd
pd.read_sql("SELECT version(), current_database(), current_user;", engine)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 7.3 MB/s eta 0:00:00


,version,current_database,current_user
0,"PostgreSQL 18.4 on x86_64-pc-linux-gnu, compil...",h5n1,h5n1_app


# Neural Network Model

**Purpose:**

We have established that the baseline model will be a random forest and the set of features that will be used.
In the milestone document, our group mentioned the use of a Long Short-Term Memory (LSTM) network, which has great capability to model sequential time-series data.
We also proposed the use of a transformer-based prediction to incorporate multiple streams of data to predict the outbreak.
The purpose of this notebook is to answer the following questions:

* 1. Does a neural network model perform better than a random forest baseline model using the same feature set (without social media features)?

* 2. What type of neural network (LSTM or transformer-based or RNN) predicts with the highest accuracy for the positive case?

# Transformer Architecture

We will be following the architecture from the Wu et al. (2025) paper that applied deep learning to predict COVID-19 cases. https://www.sciencedirect.com/science/article/pii/S1755436525000532?via%3Dihub

![Wu_et_al_transformer_architecture.jpg](attachment:fdbe18a7-b575-483f-8c8e-e57fee6f627a.jpg)


**Input Embedding**
The model first embeds the input features, case and death count data, Twitter data, county census data, and other features in the embedding step. \
They use CNN to process the sentiment analysis of Twitter before embedding it.
They end up projecting the features into a 512-dimensional embedding space.

**Positional Encoding**
Next, they pass through the positional encoder in order to embed the order of information into the data, allowing them to understand the time flow within the input. \
The mathematical relationship that they model is $y_{(t+k)},...,y_t = f_k(x_{(t-w)},...,x_{(t-1)})$ .For our work, the k = 7.

**Why they decide to not include the decoder?**
They decided to remove the decoder because it results in "simplifying the model, reducing training time, and improving performance by emphasizing direct representation learning over autoregressive decoding." \
The features fed to the model are also processed; the decoder is not essential and can cause potential overfitting.

**Transformer Encoder**
In the encoder, they use four transformer encoder layers before outputting a prediction. Each trasformer and encoder has attention heads and a feedforward network, which can be largely simplified as in this diagram:

Input sequence -> Transformer Encoder #1 -> Transformer Encoder #2 -> Transformer Encoder #3 -> Transformer Encoder #4 -> Prediction

**Attention Heads** There are 8 attention heads, 64-dimensional for Q, K, and V, and a dropout rate of 0.1.
The reason for having 8 attention heads is to focus on different aspects of the relationship between the features in the time-series dataset.
For every time point, the Transformer creates: $Q=XW_Q$ , $K=XW_k$ , $V=XW_v$ where \(X\) is the embedded time-series representation in the last w days. Then attention is approximately: $Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$ \
Q and K values pass through the softmax function to multiply V as a weight.

**Feedforward network**
The purpose of the feedforward network is to transform that information into a more useful representation. The feedforward dimension is 1024 dimensions, in which the feedforward network nonlinearly transforms the vector from 512 -> expands to 1024 -> nonlinear transformation -> 512 dimensions.

**Linear Projection**
the encoder's output is fed straight into a linear layer that projects it onto the k future values in a single forward pass. So the model doesn't "chain" predictions together — it outputs the whole forecast horizon at once.

## Training Data and Window Length

The study used a randomized window length that caps at a maximum 16-day lookback for daily prediction pr 5 weeks for weekly prediction.
The idea was to expose the model to varying amounts of historical context so it would generalize better and not overfit to one specific window size.
Randomized in this case means that the model might use a 5-day lookback and then another time 16 day look back.
The study noted that the weekly forecast model is better than the daily forecast model.

## Rough sketch of Our Transformer Architecture

<img src="attachment:b1535cc0-7cc7-4afd-af6f-ba6d15dd1998.png" width="700">

# Features used in our model

|Variable|RF Feature Importance|
|--------|---------------------|
|days_since_outbreak_county|0.110044|
|log_own_outbreaks_7d|0.090045|
|log_own_outbreaks_21d|0.086476|
|log_own_outbreaks_60d|0.050954|
|log_layer_inventory|0.045355|
|log_wintering_side_pressure|0.035853|
|doy_sin|0.032886|
|days_since_outbreak_neighbor|0.032494|
|log_turkey_operations|0.031496|
|log_poultry_operations|0.030688|

We will use doy_sin for a cyclical representation of day-of-year feature and convert day into number of days from the present day.max() - day to indicate for a look back window in days. \
We also have to include state and county (fips in data set) into the data too since the model needs that information.

## Instructions for importing the ML libraries

Since you're building the neural network model, sync that extra:

uv sync --extra ml

If you also need the classical-modeling stack in the same environment (e.g. comparing against the sklearn baseline in the same notebook), you can request multiple extras at once:

uv sync --extra ml --extra dev

Then launch the notebook the usual project way so it picks up the synced env:

uv run --extra dev jupyter lab

## 1. Load and Transform Data

In [2]:
# import libraries
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

# comment out for Colab run
# from h5n1.db import get_engine

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)

In [ ]:
# import data from database
QUERY = f"""
SELECT fips, day, state,
       target_outbreak_next_7d, days_since_outbreak_county,
       own_outbreaks_7d, own_outbreaks_21d, own_outbreaks_60d,
       layer_inventory, wintering_side_pressure, doy_sin,
       days_since_outbreak_neighbor, turkey_operations,
      poultry_operations
FROM feature_county_day
WHERE is_conus
  AND target_complete_7d
"""
# read sql
# comment out for Colab run
# df = pd.read_sql(QUERY, get_engine())
df = pd.read_sql(QUERY, engine)
# transform data, convert to proper data type
df["day"] = pd.to_datetime(df["day"])

In [ ]:
# calculate days_from_present
days_from_present =  (df["day"].max() - df["day"]).dt.days
df["days_from_present"] = days_from_present

In [ ]:
# handling NaN
# for days_since_outbreak_county and for days_since_outbreak_neighbor, we can fill with one thousands because 400 > max of 365
df['days_since_outbreak_county'] = df['days_since_outbreak_county'].fillna(400.0)
df['days_since_outbreak_neighbor'] = df['days_since_outbreak_neighbor'].fillna(400.0)

# for layer_inventory, the reason it is NaN due to the  deliberate government suppression. Lines 21-38 spell it out: NASS withholds county-level inventory numbers to avoid identifying a single farm's data
# data is not missing at random here
# they suppressed small farm
# we will fill with the layer_inventory each state median because sometime they omit small farm or very big farm
df['layer_inventory'] = (df.groupby('state')['layer_inventory'].transform(lambda s: s.fillna(s.median())))

In [ ]:
# log function
def log_transform(df, list_cols):
    new_log = []
    for col in list_cols:
        new_col = f"log_{col}"
        new_log.append(new_col)
        df[new_col] = np.log1p(df[col].clip(lower=0)) #prevent log of zero and negative
    return new_log

In [ ]:
# map function
def map_category(df, list_cols, list_vocabs):
    for i in range(len(list_cols)):
        map_categories = pd.Categorical(df[list_cols[i]])
        vocabs = {cat: idx for idx, cat in enumerate(map_categories.categories)}
        list_vocabs.append(vocabs)
        df[list_cols[i]] = df[list_cols[i]].map(vocabs)

In [ ]:
# transform the data: standard scaler, apply log, convert bool to float
# apply log and drop non log column
log_list = ["own_outbreaks_7d", "own_outbreaks_21d", "own_outbreaks_60d", "layer_inventory", "wintering_side_pressure", "turkey_operations", "poultry_operations"]
new_log = log_transform(df, log_list)
df = df.drop(columns = log_list)
# standard scaler
scaler = StandardScaler()
scale_list = [i for i in new_log if i not in ["fips", "state","days_from_present"]]
df.loc[:,scale_list] = scaler.fit_transform(df.loc[:,scale_list])
# convert bool to float
df["target_outbreak_next_7d"] = df["target_outbreak_next_7d"].astype(float)
# convert date to float
df["days_from_present"] = df["days_from_present"].astype(float)

# map each state and fips to a number
list_vocabs = []
map_category(df, list_cols = ["state", "fips"], list_vocabs = list_vocabs)


## Read From dataframe

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import libraries
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             precision_recall_curve, roc_auc_score, roc_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

# comment out for Colab run
# from h5n1.db import get_engine

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)

In [ ]:
file_path = '/content/drive/MyDrive/ColabNotebooks/elevenfeaturesdf.csv'
df = pd.read_csv(file_path)
df.head()

,fips,day,state,target_outbreak_next_7d,days_since_outbreak_county,doy_sin,days_since_outbreak_neighbor,days_from_present,log_own_outbreaks_7d,log_own_outbreaks_21d,log_own_outbreaks_60d,log_layer_inventory,log_wintering_side_pressure,log_turkey_operations,log_poultry_operations
0,692,2024-04-07,13,0.0,400.0,0.993390,400.0,828.0,-0.03994,-0.062949,-0.099113,0.225324,-0.361868,0.207407,0.070973
1,692,2024-04-26,13,0.0,400.0,0.903946,400.0,809.0,-0.03994,-0.062949,-0.099113,0.225324,-0.361868,0.207407,0.070973
2,692,2024-04-27,13,0.0,400.0,0.896456,400.0,808.0,-0.03994,-0.062949,-0.099113,0.225324,-0.361868,0.207407,0.070973
3,692,2024-05-27,13,0.0,400.0,0.561034,400.0,778.0,-0.03994,-0.062949,-0.099113,0.225324,-0.361868,0.207407,0.070973
4,692,2024-06-05,13,0.0,400.0,0.426674,400.0,769.0,-0.03994,-0.062949,-0.099113,0.225324,-0.361868,0.207407,0.070973


In [ ]:
# map function
def map_category(df, list_cols, list_vocabs):
    for i in range(len(list_cols)):
        map_categories = pd.Categorical(df[list_cols[i]])
        vocabs = {cat: idx for idx, cat in enumerate(map_categories.categories)}
        list_vocabs.append(vocabs)
        # df[list_cols[i]] = df[list_cols[i]].map(vocabs)

In [ ]:
# map each state and fips to a number
list_vocabs = []
map_category(df, list_cols = ["state", "fips"], list_vocabs = list_vocabs)

In [ ]:
len(list_vocabs)

2

## Seperate Train-Validate-Test

* The leakage in the target is potential and needs to begin for the test window more than 1 week or 7 days apart between the train, validation, and test sets.
* Every dataset has to have more than one year of data
* train 60%, val 20%, test is whatever left in the dataset
* Approximatly 52 weeks in one year

In [ ]:
df['days_from_present'].max(), df['days_from_present'].min()

(2386.0, 0.0)

In [ ]:
2386.0/7

340.85714285714283

In [4]:
def group_n_sort(df, group=['days_from_present', 'state', 'fips'], sort=['fips', 'days_from_present']):
    df = df.groupby(group).mean(numeric_only=True)
    df = df.sort_values(sort).reset_index()
    return df

In [5]:
def create_train_val_test(df, train_percent, val_percent, week_total, columns=None):
    #Define the length of days for train, val, and test
    max_day = week_total * 7
    print('max days', max_day)
    train_day = round(train_percent*max_day)
    val_day = round(val_percent*max_day)
    train_start = max_day
    train_stop = train_start - train_day
    val_start = train_stop - 7
    val_stop = val_start - val_day
    test_start = val_stop - 7
    test_stop = df["days_from_present"].min()
    print('train start', train_start, 'train stop', train_stop, 'validate start', val_start, 'validate stop', val_stop, 'test start', test_start, 'test stop', test_stop)
    # filter, group, sort the df
    # columns=None keeps the original 10 features; pass a list to use a different feature set
    if columns is None:
        columns = ["target_outbreak_next_7d", "fips", "state", "days_from_present", "days_since_outbreak_county","log_own_outbreaks_7d","log_own_outbreaks_21d","log_own_outbreaks_60d","log_layer_inventory","log_wintering_side_pressure","doy_sin","days_since_outbreak_neighbor","log_turkey_operations","log_poultry_operations"]
    df_train = df.loc[(df["days_from_present"] <= train_start) & (df["days_from_present"] >= train_stop),columns]
    df_train = group_n_sort(df_train)
    print(df_train["days_from_present"].max(), df_train["days_from_present"].min())
    df_val = df.loc[(df["days_from_present"] <= val_start) & (df["days_from_present"] >= val_stop),columns]
    df_val = group_n_sort(df_val)
    print(df_val["days_from_present"].max(), df_val["days_from_present"].min())
    df_test = df.loc[(df["days_from_present"] <= test_start),:]
    df_test = group_n_sort(df_test)
    print(df_test["days_from_present"].max(), df_test["days_from_present"].min())
    return df_train, df_val, df_test

In [ ]:
df_train, df_val, df_test = create_train_val_test(df, train_percent=0.50, val_percent=0.25, week_total=340)

max days 2380
train start 2380 train stop 1190 validate start 1183 validate stop 588 test start 581 test stop 0.0
2380.0 1190.0
1183.0 588.0
581.0 0.0


In [ ]:
print("train positive rate:", df_train["target_outbreak_next_7d"].mean())
print("val positive rate:", df_val["target_outbreak_next_7d"].mean())

train positive rate: 0.0013013193113948782
val positive rate: 0.0014279020151503374


In [ ]:
# df held the full multi-year table and is no longer needed once the splits
# are built -- freeing it here was one of the fixes for the memory pressure
# that stopped the last training run (system had ~1GB free out of 16GB)
del df
import gc
gc.collect()

60

# Transformer-based Batch

## Convert to Tensor


In [ ]:
class H5N1WindowDataset(torch.utils.data.Dataset):
    """
    One sample = one (county, window start) pair.
      features -> seq_len consecutive days of history, oldest to newest
      target   -> the forecast_horizon_k days immediately after that window
                  (closer to the present), each day's own target_outbreak_next_7d flag

    seq_len is kept short (paper caps lookback around 16 days) on purpose: a
    df_train/df_val/df_test split spans hundreds of days, and self-attention
    cost grows with seq_len^2, so treating a whole split as one giant lookback
    per county would be computationally infeasible. Instead we slide a short
    window across the split, so every day still gets used -- just spread
    across many samples instead of stuffed into one.

    Expects df_split to already be the output of group_n_sort (one row per
    county per day, sorted by fips then days_from_present) -- df_train/df_val/
    df_test from create_train_val_test already are.
    """
    FEATURE_COLS = [
        "days_from_present", "days_since_outbreak_county", "log_own_outbreaks_7d",
        "log_own_outbreaks_21d", "log_own_outbreaks_60d", "log_layer_inventory",
        "log_wintering_side_pressure", "doy_sin", "days_since_outbreak_neighbor",
        "log_turkey_operations", "log_poultry_operations",
    ]

    def __init__(self, df_split, seq_len=30, forecast_horizon_k=7, stride=7):
        self.seq_len = seq_len
        self.forecast_horizon_k = forecast_horizon_k

        self.county_data = []   # (feat_arr, target_arr, days_arr, state_code, fips_code)
        self.index = []         # (county_idx, window_start_position)
        self.target_weights = [] # Weights for WeightedRandomSampler

        window_len = seq_len + forecast_horizon_k

        # Collect all targets to calculate global class frequencies for weighting
        all_window_has_positive = []

        for county_idx, (fips_code, g) in enumerate(df_split.groupby("fips")):
            g = g.sort_values("days_from_present", ascending=False)  # oldest day first
            feat_arr = g[self.FEATURE_COLS].to_numpy(dtype="float32")
            target_arr = g["target_outbreak_next_7d"].to_numpy(dtype="float32")
            days_arr = g["days_from_present"].to_numpy()
            state_code = int(g["state"].iloc[0])
            self.county_data.append((feat_arr, target_arr, days_arr, state_code, int(fips_code)))

            last_start = len(g) - window_len
            for pos in range(0, last_start + 1, stride):
                # require the window to be calendar-contiguous -- a gap in
                # daily coverage would otherwise silently splice together
                # non-adjacent days
                if days_arr[pos] - days_arr[pos + window_len - 1] == window_len - 1:
                    self.index.append((county_idx, pos))
                    # Check if this window's target horizon contains any positive outbreak
                    window_target = target_arr[pos + self.seq_len : pos + self.seq_len + self.forecast_horizon_k]
                    all_window_has_positive.append(np.any(window_target > 0))

        # Calculate weights based on the overall frequency of positive windows
        num_positive_windows = sum(all_window_has_positive)
        num_negative_windows = len(all_window_has_positive) - num_positive_windows

        # Assign weights: higher weight for positive samples
        if num_positive_windows == 0:
            # If no positive samples, all weights are 1 (or handle as an error if this should not happen)
            positive_weight = 1.0
        else:
            positive_weight = num_negative_windows / num_positive_windows

        self.target_weights = torch.tensor(
            [positive_weight if is_pos else 1.0 for is_pos in all_window_has_positive],
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        county_idx, pos = self.index[i]
        feat_arr, target_arr, _, state_code, fips_code = self.county_data[county_idx]
        lookback = feat_arr[pos : pos + self.seq_len]
        target = target_arr[pos + self.seq_len : pos + self.seq_len + self.forecast_horizon_k]
        state_seq = np.full(self.seq_len, state_code, dtype="int64")
        fips_seq = np.full(self.seq_len, fips_code, dtype="int64")
        return (
            torch.from_numpy(lookback),
            torch.from_numpy(state_seq),
            torch.from_numpy(fips_seq),
            torch.from_numpy(target),
        )


In [ ]:
# quick sanity check: how many (county, window) samples does the sliding
# window actually produce for our current 30-day lookback / 7-day forecast config
preview_dataset = H5N1WindowDataset(df_train, seq_len=30, forecast_horizon_k=7, stride=7)
len(preview_dataset)

282828

## Input Embedding

In [ ]:
# input embedding class
class MultiSourceEmbedding(nn.Module):
    def __init__(self, n_feature_num, n_feature_state, n_feature_fips, embed_dimension = 512):
        super().__init__()
        #torch.nn.Linear for continuous features and torch.nn.Embedding(num_categories, embedding_dim) for categorical
        self.embedded_feature_layer = nn.Linear(n_feature_num, embed_dimension)
        self.state_embed = nn.Embedding(num_embeddings=n_feature_state, embedding_dim=embed_dimension)
        self.county_embed = nn.Embedding(num_embeddings=n_feature_fips, embedding_dim=embed_dimension)
        self.embed_dimension = embed_dimension

    def forward(self, tensor_feature_num, tensor_feature_state, tensor_feature_fips):
        # inputs arrive already shaped (batch, seq_len, ...) from the DataLoader.
        # nn.Linear/nn.Embedding both apply to the last dim and leave the
        # batch/seq_len dims alone, so no manual reshape is needed here -- the
        # old version hardcoded a fixed batch size in a .view(...) call, which
        # broke for any DataLoader batch_size other than the full dataset.
        embedded_feature = self.embedded_feature_layer(tensor_feature_num)  # (batch, seq_len, embed_dim)
        state_vecs = self.state_embed(tensor_feature_state)                 # (batch, seq_len, embed_dim)
        county_vecs = self.county_embed(tensor_feature_fips)                # (batch, seq_len, embed_dim)
        return embedded_feature + state_vecs + county_vecs

## Math behind the Positional Encoding

final[pos][dim] = fused[pos][dim] + pe[pos][dim]
* Our dimension in this case is 512
* Pos is the position in the fused embedding
* The final vector is the fused vector, adding with the Positional Encoding of that particular in position in the vector

PE(pos, 2i)   = sin(pos / 10000^(2i/d_model)) \
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

Wu et al. used and cited the positional encoding according to this paper: https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf


In [ ]:
# Positional encoding class
# Standard sinusoidal positional encoding (Vaswani et al., 2017).
# Input to the class is the dimension (512) and the max_len, which is days of the look back windows
# The output of the PE vector before adding to tensor is size (batch (row numbers), seq_len (lookback window), d_model (dimension))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int = 512, max_len: int = 20):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, : x.size(1), :]


## Transformer Encoder

* 8 attention heads
* 4 encoder layers
* 512 dimensions -> 1024 dimensions -> 512 dimensions
* You can edit the max lookback and forecast horizon

In [ ]:
class BirdFluForecaster(nn.Module):
    def __init__(
        self,
        n_feature_num: int,
        n_feature_state: int,
        n_feature_fips: int,
        d_model: int = 512,
        n_heads: int = 8,
        n_layers: int = 4,
        d_ff: int = 1024,
        dropout: float = 0.1,
        max_lookback: int = 30,
        forecast_horizon_k: int = 7,   # day look forward
    ):
        super().__init__()
        self.forecast_horizon_k = forecast_horizon_k

        # input embedding
        self.embedding = MultiSourceEmbedding(
            n_feature_num, n_feature_state, n_feature_fips, d_model)
        # positional encoding
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_lookback)

        # encoder layer 4 layers, 8 attention heads
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Direct projection: pool encoder output over the window,
        # then map straight to k future weekly values (no decoder loop).
        self.output_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, forecast_horizon_k),
        )

    def forward(self, tensor_feature_num, tensor_feature_state, tensor_feature_fips):
        # All inputs: (batch, seq_len=w, feature_dim)
        x = self.embedding(tensor_feature_num, tensor_feature_state, tensor_feature_fips)
        x = self.pos_encoding(x)
        encoded = self.encoder(x)              # (batch, w, d_model)

        # Pool over the look-back window (mean pooling; the paper's exact
        # pooling/readout isn't fully specified, this is a reasonable choice)
        pooled = encoded.mean(dim=1)            # (batch, d_model)

        forecast = self.output_head(pooled)     # (batch, k)
        return forecast


## Output Metrics

What they used in the Wu et al is percentage agreement which has the formula of PA = min(y, y_hat) / max(y, y_hat), averaged over all predictions. To be congruent with previous metrics in Logistic regression and random forest, we going to continue with PR-AUC. What is different in the transformer model is we have a window of 30 day look back and 7 day look forward, which means there will be PR-AUC produced for each window.

Why we need to skip when True Positive is zero? \
Because average_precision_score (PR-AUC) is mathematically undefined when there are zero positives in y_true for that slice.

PR-AUC is computed from precision = TP/(TP+FP) at each threshold, summed against recall. If there are no actual positives at all:

- Recall's denominator (TP+FN) is 0 — recall is undefined (0/0), so there's no PR curve to draw at all.
- sklearn's average_precision_score doesn't raise in this case — it silently returns 0.0 and (depending on version) may throw an UndefinedMetricWarning. That 0.0 doesn't mean "the model failed on this day"; it means "there was nothing to detect on this day," which is a completely different fact.

Recall:
PR-AUC is measures the area under the precision-recall curve across different classification thresholds. Range from 0 to 1, where 1 represents a perfect classifier and a higher score indicates better model quality.
PR-AUC lift = PR-AUC / prevalence or base rate, telling us how much better than randomly guessing. It is more comparable and interpretable than raw PR-AUC.


## Optimizing the model

* to run the smoke test  train_stride/eval_stride is bump up to 30 for example

### <u>Note of the main problem in this model</u>

* high class imbalance, we fix this by score the loss on the positive to penalize it heavily. by include this weight to the loss function: pos_weight = torch.tensor([(1 - p) / p])  # p = positive rate in train_dataset targets
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

* The pr-auc lift fluctuation from epoch to epoch due to rare to find positive in the dataset.

### <u>Hyperparameters that can be optimized</u>
* learning rate
* batch size


In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import average_precision_score

In [ ]:
def evaluate(model, loader):
    """Mean PR-AUC across the forecast horizon (skips any horizon day with
    zero positives in this loader -- PR-AUC is undefined there)."""
    model.eval()
    device = next(model.parameters()).device
    all_true, all_pred = [], []
    with torch.no_grad():
        for feat, state, fips, target in loader:
            feat, state, fips = feat.to(device), state.to(device), fips.to(device)
            logits = model(feat, state, fips)
            all_true.append(target)
            all_pred.append(logits.cpu())
    y_true = torch.cat(all_true).numpy()
    y_pred = torch.cat(all_pred).numpy()  # raw logits are fine -- PR-AUC is rank-based

    per_horizon = []
    for j in range(y_true.shape[1]):
        pos = y_true[:, j].sum()
        if pos == 0:
            continue
        ap = average_precision_score(y_true[:, j], y_pred[:, j])
        base = pos / len(y_true)          # no-skill PR-AUC for THIS horizon day
        per_horizon.append({"h": j, "pr_auc": ap, "base": base, "lift": ap / base})

    mean_pr_auc = np.mean([d["pr_auc"] for d in per_horizon])
    mean_base   = np.mean([d["base"]   for d in per_horizon])
    mean_lift   = np.mean([d["lift"]   for d in per_horizon])   # not mean_pr_auc / mean_base
    return mean_pr_auc, mean_base, mean_lift, per_horizon


In [ ]:
def train_model(
    seq_len: int = 30,
    forecast_horizon_k: int = 7,
    train_stride: int = 7,   # bigger stride -> fewer, less redundant training windows
    eval_stride: int = 1,    # dense stride for val -> a better PR-AUC estimate on a rare event
    epochs: int = 5,
    batch_size: int = 32,
    lr: float = 1e-4,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    train_dataset = H5N1WindowDataset(df_train, seq_len=seq_len, forecast_horizon_k=forecast_horizon_k, stride=train_stride)
    val_dataset = H5N1WindowDataset(df_val, seq_len=seq_len, forecast_horizon_k=forecast_horizon_k, stride=eval_stride)

    # shuffling here only reorders windows *within* train -- df_train/df_val/df_test
    # already have the 7-day purge gap between them, so this can't leak across splits
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = BirdFluForecaster(
        n_feature_num=len(H5N1WindowDataset.FEATURE_COLS),
        n_feature_state=len(list_vocabs[0]),
        n_feature_fips=len(list_vocabs[1]),
        max_lookback=seq_len,
        forecast_horizon_k=forecast_horizon_k,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    # target_outbreak_next_7d is binary, and BirdFluForecaster's output_head
    # ends in a plain nn.Linear (no sigmoid) -- BCEWithLogitsLoss expects exactly
    # that: raw logits in, 0/1 targets out. (The pasted reference used MSELoss
    # because it was forecasting continuous case counts; ours is a classification
    # target, so cross-entropy-style loss is the correct swap, not a cosmetic one.)

    # address the severe class imbalance with penalizing the loss on positive more
    # pos_rate = df_train["target_outbreak_next_7d"].mean()
    # pos_weight = torch.tensor([(1 - pos_rate) / pos_rate], device=device)
    train_targets = torch.cat([t for *_, t in DataLoader(train_dataset, batch_size=512)], dim=0)
    pos_rate = train_targets.mean().item()
    pos_weight = torch.tensor([(1 - pos_rate) / pos_rate], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

    best_pr_auc = -1
    best_state = None
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for feat, state, fips, target in train_loader:
            feat, state, fips, target = (
                feat.to(device), state.to(device), fips.to(device), target.to(device)
            )
            optimizer.zero_grad()
            logits = model(feat, state, fips)
            loss = loss_fn(logits, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        val_mean_pr_auc, val_mean_base, val_mean_lift, val_per_horizon = evaluate(model, val_loader)
        # memorize the best val pr-auc so far
        if val_mean_pr_auc > best_pr_auc:
            best_pr_auc = val_mean_pr_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        # print the loss, val pr-auc, base, and lift
        print(f"Epoch {epoch+1}/{epochs} | BCE Loss: {epoch_loss/len(train_loader):.4f} "
      f"| Val PR-AUC: {val_mean_pr_auc:.4f} | Val base rate: {val_mean_base:.4%} "
      f"| Val Lift: {val_mean_lift:.1f}x")

    model.load_state_dict(best_state)
    return model

In [ ]:
 # smoke test
 train_model(epochs=1, train_stride = 30, eval_stride = 30)

Training on device: cuda
Epoch 1/1 | BCE Loss: 1.4039 | Val PR-AUC: 0.0075 | Val base rate: 0.1523% | Val Lift: 4.9x


BirdFluForecaster(
  (embedding): MultiSourceEmbedding(
    (embedded_feature_layer): Linear(in_features=11, out_features=512, bias=True)
    (state_embed): Embedding(49, 512)
    (county_embed): Embedding(3108, 512)
  )
  (pos_encoding): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (output_head): Sequential(
    

In [ ]:
# train_stride 7 x 2, and eval stride 1 x 2
train_model(epochs=20, train_stride = 14, eval_stride = 2)

Training on device: cuda
Epoch 1/20 | BCE Loss: 1.4138 | Val PR-AUC: 0.0035 | Val base rate: 0.1569% | Val Lift: 2.2x
Epoch 2/20 | BCE Loss: 1.3837 | Val PR-AUC: 0.0016 | Val base rate: 0.1569% | Val Lift: 1.0x
Epoch 3/20 | BCE Loss: 1.4047 | Val PR-AUC: 0.0061 | Val base rate: 0.1569% | Val Lift: 3.9x
Epoch 4/20 | BCE Loss: 1.3869 | Val PR-AUC: 0.0016 | Val base rate: 0.1569% | Val Lift: 1.0x
Epoch 5/20 | BCE Loss: 1.3979 | Val PR-AUC: 0.0174 | Val base rate: 0.1569% | Val Lift: 11.2x
Epoch 6/20 | BCE Loss: 1.3877 | Val PR-AUC: 0.0174 | Val base rate: 0.1569% | Val Lift: 11.1x
Epoch 7/20 | BCE Loss: 1.3887 | Val PR-AUC: 0.0099 | Val base rate: 0.1569% | Val Lift: 6.4x
Epoch 8/20 | BCE Loss: 1.3889 | Val PR-AUC: 0.0205 | Val base rate: 0.1569% | Val Lift: 13.1x
Epoch 9/20 | BCE Loss: 1.3979 | Val PR-AUC: 0.0062 | Val base rate: 0.1569% | Val Lift: 4.0x
Epoch 10/20 | BCE Loss: 1.3872 | Val PR-AUC: 0.0224 | Val base rate: 0.1569% | Val Lift: 14.3x
Epoch 11/20 | BCE Loss: 1.3907 | Val PR-

BirdFluForecaster(
  (embedding): MultiSourceEmbedding(
    (embedded_feature_layer): Linear(in_features=11, out_features=512, bias=True)
    (state_embed): Embedding(49, 512)
    (county_embed): Embedding(3108, 512)
  )
  (pos_encoding): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (output_head): Sequential(
    

In [ ]:
# train_stride 7 x 1, and eval stride 1 x 1
best_model = train_model(epochs=20)

Training on device: cuda
Epoch 1/20 | BCE Loss: 1.4034 | Val PR-AUC: 0.0112 | Val base rate: 0.1571% | Val Lift: 7.1x
Epoch 2/20 | BCE Loss: 1.3867 | Val PR-AUC: 0.0119 | Val base rate: 0.1571% | Val Lift: 7.5x
Epoch 3/20 | BCE Loss: 1.4003 | Val PR-AUC: 0.0133 | Val base rate: 0.1571% | Val Lift: 8.5x
Epoch 4/20 | BCE Loss: 1.3865 | Val PR-AUC: 0.0180 | Val base rate: 0.1571% | Val Lift: 11.5x
Epoch 5/20 | BCE Loss: 1.4080 | Val PR-AUC: 0.0054 | Val base rate: 0.1571% | Val Lift: 3.4x
Epoch 6/20 | BCE Loss: 1.3882 | Val PR-AUC: 0.0143 | Val base rate: 0.1571% | Val Lift: 9.2x
Epoch 7/20 | BCE Loss: 1.3925 | Val PR-AUC: 0.0016 | Val base rate: 0.1571% | Val Lift: 1.0x
Epoch 8/20 | BCE Loss: 1.3997 | Val PR-AUC: 0.0088 | Val base rate: 0.1571% | Val Lift: 5.7x
Epoch 9/20 | BCE Loss: 1.3896 | Val PR-AUC: 0.0093 | Val base rate: 0.1571% | Val Lift: 6.0x
Epoch 10/20 | BCE Loss: 1.3791 | Val PR-AUC: 0.0067 | Val base rate: 0.1571% | Val Lift: 4.2x
Epoch 11/20 | BCE Loss: 1.2734 | Val PR-AUC

## Comparing train and eval stride

|Epochs|Train Stride|Eval Stride|Best Val lift|
|------|------------|-----------|-------------|
|20|7|1|27.7x|
|20|14|2|23.6x|


## Weighted Random Sampler and Increase batch

* The logic added to the H5N1WindowDataset for .target_weights function is window has no positive, it will be weight 1. If there is positive it will be weight number of negative / number of positive.


In [ ]:
from torch.utils.data import WeightedRandomSampler

In [ ]:
def train_model(
    seq_len: int = 30,
    forecast_horizon_k: int = 7,
    train_stride: int = 7,
    eval_stride: int = 1,
    epochs: int = 5,
    batch_size: int = 32,
    lr: float = 1e-4,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    train_dataset = H5N1WindowDataset(df_train, seq_len=seq_len, forecast_horizon_k=forecast_horizon_k, stride=train_stride)
    val_dataset = H5N1WindowDataset(df_val, seq_len=seq_len, forecast_horizon_k=forecast_horizon_k, stride=eval_stride)

    # Add the Weighted Random Sampler to the DataLoader
    # We also edit H5N1WindowDataset to have .target_weights function
    train_sampler = WeightedRandomSampler(
        weights=train_dataset.target_weights,
        num_samples=len(train_dataset),
        replacement=True # True for allow oversampling
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = BirdFluForecaster(
        n_feature_num=len(H5N1WindowDataset.FEATURE_COLS),
        n_feature_state=len(list_vocabs[0]),
        n_feature_fips=len(list_vocabs[1]),
        max_lookback=seq_len,
        forecast_horizon_k=forecast_horizon_k,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    train_targets = torch.cat([t for *_, t in DataLoader(train_dataset, batch_size=512)], dim=0)
    pos_rate = train_targets.mean().item()
    pos_weight = torch.tensor([(1 - pos_rate) / pos_rate], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

    best_pr_auc = -1
    best_state = None
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for feat, state, fips, target in train_loader:
            feat, state, fips, target = (
                feat.to(device), state.to(device), fips.to(device), target.to(device)
            )
            optimizer.zero_grad()
            logits = model(feat, state, fips)
            loss = loss_fn(logits, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        val_mean_pr_auc, val_mean_base, val_mean_lift, val_per_horizon = evaluate(model, val_loader)
        # memorize the best val pr-auc so far
        if val_mean_pr_auc > best_pr_auc:
            best_pr_auc = val_mean_pr_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        # print the loss, val pr-auc, base, and lift
        print(f"Epoch {epoch+1}/{epochs} | BCE Loss: {epoch_loss/len(train_loader):.4f} "
      f"| Val PR-AUC: {val_mean_pr_auc:.4f} | Val base rate: {val_mean_base:.4%} "
      f"| Val Lift: {val_mean_lift:.1f}x")

    model.load_state_dict(best_state)
    return model

In [ ]:
# weighted sample with look back 30 and forecast 7 and batch 32 lr 1e-4
best_model_ws_30_7 = train_model(epochs=10)

Training on device: cuda
Epoch 1/10 | BCE Loss: 4.2043 | Val PR-AUC: 0.0204 | Val base rate: 0.1571% | Val Lift: 13.0x
Epoch 2/10 | BCE Loss: 4.0715 | Val PR-AUC: 0.0049 | Val base rate: 0.1571% | Val Lift: 3.1x
Epoch 3/10 | BCE Loss: 4.0608 | Val PR-AUC: 0.0102 | Val base rate: 0.1571% | Val Lift: 6.5x
Epoch 4/10 | BCE Loss: 4.1878 | Val PR-AUC: 0.0039 | Val base rate: 0.1571% | Val Lift: 2.5x
Epoch 5/10 | BCE Loss: 4.1672 | Val PR-AUC: 0.0249 | Val base rate: 0.1571% | Val Lift: 15.8x
Epoch 6/10 | BCE Loss: 4.1081 | Val PR-AUC: 0.0103 | Val base rate: 0.1571% | Val Lift: 6.6x
Epoch 7/10 | BCE Loss: 4.1367 | Val PR-AUC: 0.0255 | Val base rate: 0.1571% | Val Lift: 16.3x
Epoch 8/10 | BCE Loss: 4.1327 | Val PR-AUC: 0.0077 | Val base rate: 0.1571% | Val Lift: 4.9x
Epoch 9/10 | BCE Loss: 4.1716 | Val PR-AUC: 0.0110 | Val base rate: 0.1571% | Val Lift: 7.0x
Epoch 10/10 | BCE Loss: 4.1125 | Val PR-AUC: 0.0225 | Val base rate: 0.1571% | Val Lift: 14.3x


In [ ]:
# weighted sample with look back 30 and forecast 7 batch 64 (32x2)
best_model_ws_30_7_batch64 = train_model(epochs=10, batch_size=64)

Training on device: cuda
Epoch 1/10 | BCE Loss: 4.2604 | Val PR-AUC: 0.0372 | Val base rate: 0.1571% | Val Lift: 23.8x
Epoch 2/10 | BCE Loss: 4.0893 | Val PR-AUC: 0.0180 | Val base rate: 0.1571% | Val Lift: 11.5x
Epoch 3/10 | BCE Loss: 4.1530 | Val PR-AUC: 0.0078 | Val base rate: 0.1571% | Val Lift: 5.0x
Epoch 4/10 | BCE Loss: 4.2941 | Val PR-AUC: 0.0016 | Val base rate: 0.1571% | Val Lift: 1.0x
Epoch 5/10 | BCE Loss: 4.2058 | Val PR-AUC: 0.0138 | Val base rate: 0.1571% | Val Lift: 8.8x
Epoch 6/10 | BCE Loss: 4.0568 | Val PR-AUC: 0.0248 | Val base rate: 0.1571% | Val Lift: 15.8x
Epoch 7/10 | BCE Loss: 4.1298 | Val PR-AUC: 0.0062 | Val base rate: 0.1571% | Val Lift: 3.9x


In [ ]:
# weighted sample with look back 30 and forecast 7 batch 128 (32x4) double lr 2e-4
best_model_ws_30_7_batch128 = train_model(epochs=5, batch_size=128, lr=2e-4)

Training on device: cuda
Epoch 1/5 | BCE Loss: 4.2367 | Val PR-AUC: 0.0290 | Val base rate: 0.1571% | Val Lift: 18.5x
Epoch 2/5 | BCE Loss: 4.0526 | Val PR-AUC: 0.0154 | Val base rate: 0.1571% | Val Lift: 9.9x
Epoch 3/5 | BCE Loss: 4.1956 | Val PR-AUC: 0.0410 | Val base rate: 0.1571% | Val Lift: 26.2x
Epoch 4/5 | BCE Loss: 4.1000 | Val PR-AUC: 0.0224 | Val base rate: 0.1571% | Val Lift: 14.3x
Epoch 5/5 | BCE Loss: 4.0527 | Val PR-AUC: 0.0403 | Val base rate: 0.1571% | Val Lift: 25.6x


In [ ]:
# weighted sample with look back 30 and forecast 7 batch 128 (32x4) double lr 2e-4
best_model_ws_30_7_batch256 = train_model(epochs=5, batch_size=256, lr=2e-4)

Training on device: cuda
Epoch 1/5 | BCE Loss: 4.4393 | Val PR-AUC: 0.0096 | Val base rate: 0.1571% | Val Lift: 6.1x
Epoch 2/5 | BCE Loss: 4.0945 | Val PR-AUC: 0.0214 | Val base rate: 0.1571% | Val Lift: 13.7x
Epoch 3/5 | BCE Loss: 4.0519 | Val PR-AUC: 0.0278 | Val base rate: 0.1571% | Val Lift: 17.7x
Epoch 4/5 | BCE Loss: 4.0480 | Val PR-AUC: 0.0034 | Val base rate: 0.1571% | Val Lift: 2.2x
Epoch 5/5 | BCE Loss: 4.0815 | Val PR-AUC: 0.0078 | Val base rate: 0.1571% | Val Lift: 5.0x


In [ ]:
# weighted sample with look back 30 and forecast 7 batch 128 (32x4) quaduple lr 4e-4
best_model_ws_30_7_batch256_4e_4 = train_model(epochs=5, batch_size=256, lr=4e-4)

Training on device: cuda
Epoch 1/5 | BCE Loss: 4.3859 | Val PR-AUC: 0.0232 | Val base rate: 0.1571% | Val Lift: 14.8x
Epoch 2/5 | BCE Loss: 4.1818 | Val PR-AUC: 0.0024 | Val base rate: 0.1571% | Val Lift: 1.5x
Epoch 3/5 | BCE Loss: 4.3299 | Val PR-AUC: 0.0022 | Val base rate: 0.1571% | Val Lift: 1.4x
Epoch 4/5 | BCE Loss: 4.3288 | Val PR-AUC: 0.0050 | Val base rate: 0.1571% | Val Lift: 3.2x
Epoch 5/5 | BCE Loss: 4.3319 | Val PR-AUC: 0.0034 | Val base rate: 0.1571% | Val Lift: 2.2x


In [ ]:
# weighted sample with look back 30 and forecast 7 batch 128 (32x4) double lr 2e-4
best_model_ws_30_7_batch128 = train_model(epochs=10, batch_size=128, lr=2e-4)

Training on device: cuda
Epoch 1/10 | BCE Loss: 4.2936 | Val PR-AUC: 0.0009 | Val base rate: 0.1571% | Val Lift: 0.6x
Epoch 2/10 | BCE Loss: 4.1171 | Val PR-AUC: 0.0231 | Val base rate: 0.1571% | Val Lift: 14.8x
Epoch 3/10 | BCE Loss: 4.1746 | Val PR-AUC: 0.0027 | Val base rate: 0.1571% | Val Lift: 1.7x
Epoch 4/10 | BCE Loss: 4.1538 | Val PR-AUC: 0.0335 | Val base rate: 0.1571% | Val Lift: 21.3x
Epoch 5/10 | BCE Loss: 4.1228 | Val PR-AUC: 0.0128 | Val base rate: 0.1571% | Val Lift: 8.1x
Epoch 6/10 | BCE Loss: 4.1247 | Val PR-AUC: 0.0014 | Val base rate: 0.1571% | Val Lift: 0.9x
Epoch 7/10 | BCE Loss: 4.0782 | Val PR-AUC: 0.0216 | Val base rate: 0.1571% | Val Lift: 13.8x
Epoch 8/10 | BCE Loss: 3.9832 | Val PR-AUC: 0.0440 | Val base rate: 0.1571% | Val Lift: 28.1x
Epoch 9/10 | BCE Loss: 3.9668 | Val PR-AUC: 0.0263 | Val base rate: 0.1571% | Val Lift: 16.8x
Epoch 10/10 | BCE Loss: 4.1852 | Val PR-AUC: 0.0041 | Val base rate: 0.1571% | Val Lift: 2.6x


|lr|Batch|Mean Val lift|
|--|------|-------------|
|1e-4|32|9x|
|1e-4|64|9.9x|
|2e-4|128|13.5x|
|2e-4|256|8.94x|
|4e-4|256|4.62|

lr 2e-4 and batch 128 currently give the best lift

## Reduce the over correctness of the imbalance dataset

In [ ]:
def train_model(
    seq_len: int = 30,
    forecast_horizon_k: int = 7,
    train_stride: int = 7,
    eval_stride: int = 1,
    epochs: int = 5,
    batch_size: int = 128,
    lr: float = 2e-4,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    train_dataset = H5N1WindowDataset(df_train, seq_len=seq_len, forecast_horizon_k=forecast_horizon_k, stride=train_stride)
    val_dataset = H5N1WindowDataset(df_val, seq_len=seq_len, forecast_horizon_k=forecast_horizon_k, stride=eval_stride)

    # Add the Weighted Random Sampler to the DataLoader
    # We also edit H5N1WindowDataset to have .target_weights function
    train_sampler = WeightedRandomSampler(
        weights=train_dataset.target_weights,
        num_samples=len(train_dataset),
        replacement=True # True for allow oversampling
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=train_sampler)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = BirdFluForecaster(
        n_feature_num=len(H5N1WindowDataset.FEATURE_COLS),
        n_feature_state=len(list_vocabs[0]),
        n_feature_fips=len(list_vocabs[1]),
        max_lookback=seq_len,
        forecast_horizon_k=forecast_horizon_k,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    loss_fn = nn.BCEWithLogitsLoss()

    best_pr_auc = -1
    best_state = None
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for feat, state, fips, target in train_loader:
            feat, state, fips, target = (
                feat.to(device), state.to(device), fips.to(device), target.to(device)
            )
            optimizer.zero_grad()
            logits = model(feat, state, fips)
            loss = loss_fn(logits, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        val_mean_pr_auc, val_mean_base, val_mean_lift, val_per_horizon = evaluate(model, val_loader)
        # memorize the best val pr-auc so far
        if val_mean_pr_auc > best_pr_auc:
            best_pr_auc = val_mean_pr_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        # print the loss, val pr-auc, base, and lift
        print(f"Epoch {epoch+1}/{epochs} | BCE Loss: {epoch_loss/len(train_loader):.4f} "
      f"| Val PR-AUC: {val_mean_pr_auc:.4f} | Val base rate: {val_mean_base:.4%} "
      f"| Val Lift: {val_mean_lift:.1f}x")

    model.load_state_dict(best_state)
    return model

In [ ]:
# weighted sample with look back 30 and forecast 7 batch 128 (32x4) double lr 2e-4 and normal loss
best_model_ws_30_7_batch128 = train_model(epochs=10)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.5386 | Val PR-AUC: 0.0152 | Val base rate: 0.1571% | Val Lift: 9.8x
Epoch 2/10 | BCE Loss: 0.5203 | Val PR-AUC: 0.0189 | Val base rate: 0.1571% | Val Lift: 12.1x
Epoch 3/10 | BCE Loss: 0.5020 | Val PR-AUC: 0.0269 | Val base rate: 0.1571% | Val Lift: 17.2x
Epoch 4/10 | BCE Loss: 0.4774 | Val PR-AUC: 0.0292 | Val base rate: 0.1571% | Val Lift: 18.6x
Epoch 5/10 | BCE Loss: 0.4616 | Val PR-AUC: 0.0279 | Val base rate: 0.1571% | Val Lift: 17.8x
Epoch 6/10 | BCE Loss: 0.4447 | Val PR-AUC: 0.0233 | Val base rate: 0.1571% | Val Lift: 14.9x
Epoch 7/10 | BCE Loss: 0.4250 | Val PR-AUC: 0.0316 | Val base rate: 0.1571% | Val Lift: 20.1x
Epoch 8/10 | BCE Loss: 0.4050 | Val PR-AUC: 0.0238 | Val base rate: 0.1571% | Val Lift: 15.2x
Epoch 9/10 | BCE Loss: 0.3851 | Val PR-AUC: 0.0204 | Val base rate: 0.1571% | Val Lift: 12.9x
Epoch 10/10 | BCE Loss: 0.3700 | Val PR-AUC: 0.0236 | Val base rate: 0.1571% | Val Lift: 15.0x


## Increase the Val PR-AUC
Now that the Val result is stabilize. We will aim to increase the val PR-AUC.

In [ ]:
#lr 4e-4
best_model_ws_30_7_batch128_4e_4 = train_model(lr=4e-4)

Training on device: cuda
Epoch 1/5 | BCE Loss: 0.5637 | Val PR-AUC: 0.0173 | Val base rate: 0.1571% | Val Lift: 11.0x
Epoch 2/5 | BCE Loss: 0.6099 | Val PR-AUC: 0.0036 | Val base rate: 0.1571% | Val Lift: 2.3x
Epoch 3/5 | BCE Loss: 0.6099 | Val PR-AUC: 0.0123 | Val base rate: 0.1571% | Val Lift: 7.9x
Epoch 4/5 | BCE Loss: 0.5424 | Val PR-AUC: 0.0207 | Val base rate: 0.1571% | Val Lift: 13.2x
Epoch 5/5 | BCE Loss: 0.5238 | Val PR-AUC: 0.0152 | Val base rate: 0.1571% | Val Lift: 9.7x


In [ ]:
#seq len 60
best_model_ws_60_7_batch128_2e_4 = train_model(seq_len=60)

Training on device: cuda
Epoch 1/5 | BCE Loss: 0.5163 | Val PR-AUC: 0.0226 | Val base rate: 0.1791% | Val Lift: 12.6x
Epoch 2/5 | BCE Loss: 0.5004 | Val PR-AUC: 0.0559 | Val base rate: 0.1791% | Val Lift: 31.4x
Epoch 3/5 | BCE Loss: 0.4908 | Val PR-AUC: 0.0202 | Val base rate: 0.1791% | Val Lift: 11.4x
Epoch 4/5 | BCE Loss: 0.4774 | Val PR-AUC: 0.0356 | Val base rate: 0.1791% | Val Lift: 20.0x
Epoch 5/5 | BCE Loss: 0.4639 | Val PR-AUC: 0.0272 | Val base rate: 0.1791% | Val Lift: 15.3x


In [ ]:
#seq len 90
best_model_ws_90_7_batch128_2e_4 = train_model(seq_len=90)

Training on device: cuda
Epoch 1/5 | BCE Loss: 0.5262 | Val PR-AUC: 0.0389 | Val base rate: 0.2190% | Val Lift: 17.8x
Epoch 2/5 | BCE Loss: 0.5120 | Val PR-AUC: 0.0065 | Val base rate: 0.2190% | Val Lift: 3.0x
Epoch 3/5 | BCE Loss: 0.5023 | Val PR-AUC: 0.0089 | Val base rate: 0.2190% | Val Lift: 4.1x
Epoch 4/5 | BCE Loss: 0.4971 | Val PR-AUC: 0.0070 | Val base rate: 0.2190% | Val Lift: 3.2x
Epoch 5/5 | BCE Loss: 0.4926 | Val PR-AUC: 0.0048 | Val base rate: 0.2190% | Val Lift: 2.2x


# Simple Neural Network (MLP) baseline

Before tuning the transformer further, it helps to know what a *plain* neural network can do with the
same features. This answers question 1 of this notebook more directly: the random forest sees **one row
per county per day**, and so does this model. That makes the comparison apples-to-apples. The transformer
also changes the input shape (30-day windows) and the output (7 horizon days), which the RF doesn't.

**What is an MLP (multi-layer perceptron)?** It's the simplest neural network: a stack of fully
connected (`nn.Linear`) layers with a non-linearity (`ReLU`) between them. There's no notion of time or
sequence. Each county-day row goes in on its own and one number comes out: the logit (log-odds) that an
outbreak happens in that county in the next 7 days.

```
10 features -> Linear(10, 64) -> ReLU -> Dropout -> Linear(64, 32) -> ReLU -> Dropout -> Linear(32, 1) -> logit
```

**Design choices:**
* **Same target as the RF:** `target_outbreak_next_7d`, one label per row, so a single output.
* **`days_from_present` is dropped.** It's a time index, and because the split is by time, the test set's
  values (0–238) are never seen in training (448–1120). A per-row model would just be extrapolating a trend.
  The transformer only uses it *within* a window, which is a different situation.
* **Features are re-standardized on the training split only.** `days_since_outbreak_county` and
  `days_since_outbreak_neighbor` are still on a raw 0–400 scale. Neural nets train badly when one input is
  hundreds of times larger than the others. Fitting the scaler on train only keeps val and test statistics
  out of training.
* **Loss:** `BCEWithLogitsLoss`, the same loss the transformer uses. It takes raw logits and applies the
  sigmoid internally, which is more numerically stable than a `Sigmoid` layer followed by `BCELoss`.
* **Metric:** PR-AUC and lift on validation, the same as above. Every epoch keeps the weights with the best
  val PR-AUC (early stopping by checkpoint).


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import average_precision_score

MLP_FEATURES = [
    "days_since_outbreak_county", "log_own_outbreaks_7d", "log_own_outbreaks_21d",
    "log_own_outbreaks_60d", "log_layer_inventory", "log_wintering_side_pressure",
    "doy_sin", "days_since_outbreak_neighbor", "log_turkey_operations", "log_poultry_operations",
]
TARGET = "target_outbreak_next_7d"

# fit on train only -- val/test are transformed with the train mean/std
mlp_scaler = StandardScaler().fit(df_train[MLP_FEATURES])

def to_tensors(df_split):
    X = torch.tensor(mlp_scaler.transform(df_split[MLP_FEATURES]), dtype=torch.float32)
    # group_n_sort averaged rows, so force the label back to a clean 0/1
    y = torch.tensor((df_split[TARGET] > 0).to_numpy(), dtype=torch.float32).unsqueeze(1)  # (n, 1)
    return X, y

X_train, y_train = to_tensors(df_train)
X_val, y_val = to_tensors(df_val)
X_test, y_test = to_tensors(df_test)

for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:5s} rows: {len(X):>9,} | positives: {int(y.sum()):>5,} | positive rate: {y.mean().item():.4%}")


NameError: name 'df_train' is not defined

In [6]:
class SimpleMLP(nn.Module):
    def __init__(self, n_features, hidden=(64, 32), dropout=0.2):
        super().__init__()
        layers, prev = [], n_features
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))  # one logit -- no sigmoid, BCEWithLogitsLoss handles it
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)  # (batch, 1)


## Train the MLP

* **Batch size 1024.** With a positive rate of about 0.2%, a batch of 32 almost never contains a positive,
  so most gradient steps would only learn "predict 0". Rows are also cheap for an MLP, so larger batches are
  fine.
* **`pos_weight` is off by default.** The transformer experiments above showed that the full
  `neg/pos` weight (~430) made training unstable. PR-AUC only depends on how rows are *ranked*, so the model
  doesn't need to be pushed toward predicting positives to score well. Try a moderate value (e.g. 10) as an
  experiment below.
* Prediction runs in chunks (`predict_logits`) so val/test never have to fit on the GPU all at once.


In [7]:
def predict_logits(model, X, batch_size=8192):
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        out = [model(X[i:i + batch_size].to(device)).cpu() for i in range(0, len(X), batch_size)]
    return torch.cat(out).squeeze(1).numpy()


def pr_auc_lift(y, scores):
    y = np.asarray(y).ravel()
    ap = average_precision_score(y, scores)  # rank-based, raw logits are fine
    base = y.mean()                          # no-skill PR-AUC = positive rate
    return ap, base, ap / base


def train_mlp(
    hidden=(64, 32),
    dropout: float = 0.2,
    epochs: int = 10,
    batch_size: int = 1024,
    lr: float = 1e-3,
    pos_weight: float = None,
    seed: int = 0,
):
    torch.manual_seed(seed)  # makes runs comparable when changing one hyperparameter
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

    model = SimpleMLP(X_train.shape[1], hidden=hidden, dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    pw = None if pos_weight is None else torch.tensor([pos_weight], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)

    best_pr_auc = -1
    best_state = None
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        val_scores = predict_logits(model, X_val)
        val_pr_auc, val_base, val_lift = pr_auc_lift(y_val, val_scores)
        val_roc_auc = roc_auc_score(y_val.numpy().ravel(), val_scores)
        if val_pr_auc > best_pr_auc:
            best_pr_auc = val_pr_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"Epoch {epoch+1}/{epochs} | BCE Loss: {epoch_loss/len(train_loader):.4f} "
              f"| Val PR-AUC: {val_pr_auc:.4f} | Val base rate: {val_base:.4%} | Val Lift: {val_lift:.1f}x "
              f"| Val ROC-AUC: {val_roc_auc:.4f}")

    model.load_state_dict(best_state)
    print(f"Best val PR-AUC: {best_pr_auc:.4f}")
    return model


In [ ]:
mlp_model = train_mlp(epochs=10)


Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0142 | Val PR-AUC: 0.0565 | Val base rate: 0.1428% | Val Lift: 39.6x | Val ROC-AUC: 0.8188
Epoch 2/10 | BCE Loss: 0.0079 | Val PR-AUC: 0.0619 | Val base rate: 0.1428% | Val Lift: 43.4x | Val ROC-AUC: 0.8340
Epoch 3/10 | BCE Loss: 0.0077 | Val PR-AUC: 0.0649 | Val base rate: 0.1428% | Val Lift: 45.4x | Val ROC-AUC: 0.8295
Epoch 4/10 | BCE Loss: 0.0076 | Val PR-AUC: 0.0616 | Val base rate: 0.1428% | Val Lift: 43.1x | Val ROC-AUC: 0.8322
Epoch 5/10 | BCE Loss: 0.0076 | Val PR-AUC: 0.0609 | Val base rate: 0.1428% | Val Lift: 42.6x | Val ROC-AUC: 0.8371
Epoch 6/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0645 | Val base rate: 0.1428% | Val Lift: 45.1x | Val ROC-AUC: 0.8375
Epoch 7/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0639 | Val base rate: 0.1428% | Val Lift: 44.7x | Val ROC-AUC: 0.8413
Epoch 8/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0645 | Val base rate: 0.1428% | Val Lift: 45.2x | Val ROC-AUC: 0.8407
Epoch 9/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0

## Experiment: learning rate

In [ ]:
mlp_model_lr_1e2 = train_mlp(epochs=10, lr=1e-2)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0091 | Val PR-AUC: 0.0378 | Val base rate: 0.1428% | Val Lift: 26.5x | Val ROC-AUC: 0.8104
Epoch 2/10 | BCE Loss: 0.0081 | Val PR-AUC: 0.0472 | Val base rate: 0.1428% | Val Lift: 33.1x | Val ROC-AUC: 0.8366
Epoch 3/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0594 | Val base rate: 0.1428% | Val Lift: 41.6x | Val ROC-AUC: 0.8208
Epoch 4/10 | BCE Loss: 0.0079 | Val PR-AUC: 0.0621 | Val base rate: 0.1428% | Val Lift: 43.5x | Val ROC-AUC: 0.8189
Epoch 5/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0688 | Val base rate: 0.1428% | Val Lift: 48.1x | Val ROC-AUC: 0.8363
Epoch 6/10 | BCE Loss: 0.0079 | Val PR-AUC: 0.0628 | Val base rate: 0.1428% | Val Lift: 44.0x | Val ROC-AUC: 0.8288
Epoch 7/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0696 | Val base rate: 0.1428% | Val Lift: 48.8x | Val ROC-AUC: 0.8397
Epoch 8/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0682 | Val base rate: 0.1428% | Val Lift: 47.8x | Val ROC-AUC: 0.8309
Epoch 9/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0

In [ ]:
mlp_model_lr_1e1 = train_mlp(epochs=10, lr=1e-1)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0103 | Val PR-AUC: 0.0016 | Val base rate: 0.1428% | Val Lift: 1.1x | Val ROC-AUC: 0.5571
Epoch 2/10 | BCE Loss: 0.0127 | Val PR-AUC: 0.0014 | Val base rate: 0.1428% | Val Lift: 1.0x | Val ROC-AUC: 0.5026
Epoch 3/10 | BCE Loss: 0.0136 | Val PR-AUC: 0.0014 | Val base rate: 0.1428% | Val Lift: 1.0x | Val ROC-AUC: 0.4976
Epoch 4/10 | BCE Loss: 0.0124 | Val PR-AUC: 0.0014 | Val base rate: 0.1428% | Val Lift: 1.0x | Val ROC-AUC: 0.5006
Epoch 5/10 | BCE Loss: 0.0126 | Val PR-AUC: 0.0034 | Val base rate: 0.1428% | Val Lift: 2.4x | Val ROC-AUC: 0.6398
Epoch 6/10 | BCE Loss: 0.0113 | Val PR-AUC: 0.0014 | Val base rate: 0.1428% | Val Lift: 1.0x | Val ROC-AUC: 0.5000
Epoch 7/10 | BCE Loss: 0.0112 | Val PR-AUC: 0.0014 | Val base rate: 0.1428% | Val Lift: 1.0x | Val ROC-AUC: 0.5000
Epoch 8/10 | BCE Loss: 0.0116 | Val PR-AUC: 0.0014 | Val base rate: 0.1428% | Val Lift: 1.0x | Val ROC-AUC: 0.5000
Epoch 9/10 | BCE Loss: 0.0127 | Val PR-AUC: 0.0365 | Va

In [ ]:
mlp_model_lr_1e4 = train_mlp(epochs=10, lr=1e-4)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0613 | Val PR-AUC: 0.0405 | Val base rate: 0.1428% | Val Lift: 28.4x | Val ROC-AUC: 0.7286
Epoch 2/10 | BCE Loss: 0.0089 | Val PR-AUC: 0.0525 | Val base rate: 0.1428% | Val Lift: 36.7x | Val ROC-AUC: 0.7891
Epoch 3/10 | BCE Loss: 0.0084 | Val PR-AUC: 0.0558 | Val base rate: 0.1428% | Val Lift: 39.1x | Val ROC-AUC: 0.8039
Epoch 4/10 | BCE Loss: 0.0082 | Val PR-AUC: 0.0583 | Val base rate: 0.1428% | Val Lift: 40.8x | Val ROC-AUC: 0.8140
Epoch 5/10 | BCE Loss: 0.0081 | Val PR-AUC: 0.0586 | Val base rate: 0.1428% | Val Lift: 41.1x | Val ROC-AUC: 0.8198
Epoch 6/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0609 | Val base rate: 0.1428% | Val Lift: 42.7x | Val ROC-AUC: 0.8223
Epoch 7/10 | BCE Loss: 0.0079 | Val PR-AUC: 0.0633 | Val base rate: 0.1428% | Val Lift: 44.3x | Val ROC-AUC: 0.8259
Epoch 8/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0630 | Val base rate: 0.1428% | Val Lift: 44.1x | Val ROC-AUC: 0.8292
Epoch 9/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0

### Experiment: moderate positive-class weight

This tests whether a gentle up-weighting of positives helps the ranking, compared with the unweighted run.


In [ ]:
mlp_model_pw10 = train_mlp(epochs=10, pos_weight=10.0)


Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0565 | Val PR-AUC: 0.0656 | Val base rate: 0.1428% | Val Lift: 46.0x | Val ROC-AUC: 0.8339
Epoch 2/10 | BCE Loss: 0.0473 | Val PR-AUC: 0.0661 | Val base rate: 0.1428% | Val Lift: 46.3x | Val ROC-AUC: 0.8378
Epoch 3/10 | BCE Loss: 0.0462 | Val PR-AUC: 0.0650 | Val base rate: 0.1428% | Val Lift: 45.5x | Val ROC-AUC: 0.8318
Epoch 4/10 | BCE Loss: 0.0460 | Val PR-AUC: 0.0575 | Val base rate: 0.1428% | Val Lift: 40.3x | Val ROC-AUC: 0.8352
Epoch 5/10 | BCE Loss: 0.0457 | Val PR-AUC: 0.0606 | Val base rate: 0.1428% | Val Lift: 42.5x | Val ROC-AUC: 0.8379
Epoch 6/10 | BCE Loss: 0.0454 | Val PR-AUC: 0.0630 | Val base rate: 0.1428% | Val Lift: 44.1x | Val ROC-AUC: 0.8402
Epoch 7/10 | BCE Loss: 0.0455 | Val PR-AUC: 0.0667 | Val base rate: 0.1428% | Val Lift: 46.7x | Val ROC-AUC: 0.8414
Epoch 8/10 | BCE Loss: 0.0451 | Val PR-AUC: 0.0624 | Val base rate: 0.1428% | Val Lift: 43.7x | Val ROC-AUC: 0.8412
Epoch 9/10 | BCE Loss: 0.0452 | Val PR-AUC: 0.0

In [ ]:
mlp_model_pw5 = train_mlp(epochs=10, pos_weight=5.0)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0359 | Val PR-AUC: 0.0634 | Val base rate: 0.1428% | Val Lift: 44.4x | Val ROC-AUC: 0.8312
Epoch 2/10 | BCE Loss: 0.0281 | Val PR-AUC: 0.0649 | Val base rate: 0.1428% | Val Lift: 45.5x | Val ROC-AUC: 0.8369
Epoch 3/10 | BCE Loss: 0.0274 | Val PR-AUC: 0.0632 | Val base rate: 0.1428% | Val Lift: 44.3x | Val ROC-AUC: 0.8313
Epoch 4/10 | BCE Loss: 0.0272 | Val PR-AUC: 0.0578 | Val base rate: 0.1428% | Val Lift: 40.5x | Val ROC-AUC: 0.8351
Epoch 5/10 | BCE Loss: 0.0270 | Val PR-AUC: 0.0578 | Val base rate: 0.1428% | Val Lift: 40.5x | Val ROC-AUC: 0.8383
Epoch 6/10 | BCE Loss: 0.0269 | Val PR-AUC: 0.0637 | Val base rate: 0.1428% | Val Lift: 44.6x | Val ROC-AUC: 0.8400
Epoch 7/10 | BCE Loss: 0.0269 | Val PR-AUC: 0.0657 | Val base rate: 0.1428% | Val Lift: 46.0x | Val ROC-AUC: 0.8405
Epoch 8/10 | BCE Loss: 0.0267 | Val PR-AUC: 0.0633 | Val base rate: 0.1428% | Val Lift: 44.3x | Val ROC-AUC: 0.8414
Epoch 9/10 | BCE Loss: 0.0267 | Val PR-AUC: 0.0

In [ ]:
mlp_model_pw1 = train_mlp(epochs=10, pos_weight=1.0)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0142 | Val PR-AUC: 0.0565 | Val base rate: 0.1428% | Val Lift: 39.6x | Val ROC-AUC: 0.8188
Epoch 2/10 | BCE Loss: 0.0079 | Val PR-AUC: 0.0619 | Val base rate: 0.1428% | Val Lift: 43.4x | Val ROC-AUC: 0.8340
Epoch 3/10 | BCE Loss: 0.0077 | Val PR-AUC: 0.0649 | Val base rate: 0.1428% | Val Lift: 45.4x | Val ROC-AUC: 0.8295
Epoch 4/10 | BCE Loss: 0.0076 | Val PR-AUC: 0.0616 | Val base rate: 0.1428% | Val Lift: 43.1x | Val ROC-AUC: 0.8322
Epoch 5/10 | BCE Loss: 0.0076 | Val PR-AUC: 0.0609 | Val base rate: 0.1428% | Val Lift: 42.6x | Val ROC-AUC: 0.8371
Epoch 6/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0645 | Val base rate: 0.1428% | Val Lift: 45.1x | Val ROC-AUC: 0.8375
Epoch 7/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0639 | Val base rate: 0.1428% | Val Lift: 44.7x | Val ROC-AUC: 0.8413
Epoch 8/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0645 | Val base rate: 0.1428% | Val Lift: 45.2x | Val ROC-AUC: 0.8407
Epoch 9/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0

## Hidden Layers

In [ ]:
mlp_model_layer_128_64_32 = train_mlp(hidden=(128, 64, 32))

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0122 | Val PR-AUC: 0.0622 | Val base rate: 0.1428% | Val Lift: 43.6x | Val ROC-AUC: 0.8397
Epoch 2/10 | BCE Loss: 0.0079 | Val PR-AUC: 0.0634 | Val base rate: 0.1428% | Val Lift: 44.4x | Val ROC-AUC: 0.8364
Epoch 3/10 | BCE Loss: 0.0076 | Val PR-AUC: 0.0610 | Val base rate: 0.1428% | Val Lift: 42.7x | Val ROC-AUC: 0.8323
Epoch 4/10 | BCE Loss: 0.0076 | Val PR-AUC: 0.0385 | Val base rate: 0.1428% | Val Lift: 27.0x | Val ROC-AUC: 0.8395
Epoch 5/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0644 | Val base rate: 0.1428% | Val Lift: 45.1x | Val ROC-AUC: 0.8414
Epoch 6/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0654 | Val base rate: 0.1428% | Val Lift: 45.8x | Val ROC-AUC: 0.8389
Epoch 7/10 | BCE Loss: 0.0074 | Val PR-AUC: 0.0652 | Val base rate: 0.1428% | Val Lift: 45.7x | Val ROC-AUC: 0.8326
Epoch 8/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0688 | Val base rate: 0.1428% | Val Lift: 48.2x | Val ROC-AUC: 0.8371
Epoch 9/10 | BCE Loss: 0.0074 | Val PR-AUC: 0.0

## Final check on the test set

**Run this only once, after you've finished tuning on validation.** If you keep changing the model to
improve the test score, the test set turns into a second validation set. Its number would then overstate
how the model does on truly unseen data.

Compare this against the random forest's test PR-AUC and lift from `regression_baseline.ipynb`. The
comparison is only fair if both models use the same test window.


In [ ]:
test_pr_auc, test_base, test_lift = pr_auc_lift(y_test, predict_logits(mlp_model, X_test))
print(f"MLP test PR-AUC: {test_pr_auc:.4f} | base rate: {test_base:.4%} | lift: {test_lift:.1f}x")


MLP test PR-AUC: 0.1253 | base rate: 0.2686% | lift: 46.6x


In [ ]:
test_pr_auc, test_base, test_lift = pr_auc_lift(y_test, predict_logits(mlp_model_lr_1e2, X_test))
print(f"MLP test PR-AUC: {test_pr_auc:.4f} | base rate: {test_base:.4%} | lift: {test_lift:.1f}x")

MLP test PR-AUC: 0.1245 | base rate: 0.2686% | lift: 46.4x


## Experiment with more features

### Top 22 features from the random forest

The MLP above uses the random forest's top 10 features. Here we give it the **top 22**, ranked by
`importances_forest` in `regression_baseline.ipynb` (300-tree forest, mean Gini reduction). In that
notebook's fewer-features test, the forest did best at roughly this size: importance >= 0.020
(23 features) gave lift 55.6x, compared with 55.1x for the top 10. Here we check whether the neural network also
gains from the extra 12 columns: more neighbor-infection counts, weather, population,
breeding-side migration pressure, and more flock types.

* The Drive CSV only has the original 10 features, so this section pulls the raw columns from the
  database again. **Run the connection cell at the top of the notebook first** (it creates `engine`).
  The result is cached on Drive as parquet, so the slow query only runs once.
* Preprocessing is the same as for the 10-feature version. Weather NULLs mean the station didn't
  report, not "0 degrees". They're filled with the **train-split median** *after* the split, so
  no information leaks in from val/test.
* Everything else (split dates, `train_mlp`, hyperparameters) stays the same, so any change in
  val PR-AUC comes from the features alone.

In [9]:
import os
import gc

import sqlalchemy
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

TARGET = "target_outbreak_next_7d"

# Top 22 by random-forest importance (regression_baseline.ipynb, `importances_forest`)
TOP22_FEATURES = [
    "days_since_outbreak_county",    # 0.1100
    "log_own_outbreaks_7d",          # 0.0900
    "log_own_outbreaks_21d",         # 0.0865
    "log_own_outbreaks_60d",         # 0.0510
    "log_layer_inventory",           # 0.0454
    "log_wintering_side_pressure",   # 0.0359
    "doy_sin",                       # 0.0329
    "days_since_outbreak_neighbor",  # 0.0325
    "log_turkey_operations",         # 0.0315
    "log_poultry_operations",        # 0.0307
    "log_population",                # 0.0296  <- new from here down
    "log_breeding_side_pressure",    # 0.0294
    "log_nbr_infections_7d",         # 0.0289
    "temp_max_7d_c",                 # 0.0283
    "log_nbr_infections_21d",        # 0.0260
    "temp_min_7d_c",                 # 0.0257
    "log_nbr_wild_detections_21d",   # 0.0246
    "log_broiler_operations",        # 0.0244
    "log_duck_operations",           # 0.0239
    "doy_cos",                       # 0.0220
    "log_nbr_infections_60d",        # 0.0214
    "precip_7d_mm",                  # 0.0202
]
assert len(TOP22_FEATURES) == 22

# the database stores raw counts; the log_ versions are built with log1p during the load
RAW22_COLS = [f.removeprefix("log_") for f in TOP22_FEATURES]
LOG22_COLS = [f.removeprefix("log_") for f in TOP22_FEATURES if f.startswith("log_")]


def preprocess_window(w):
    """Same preprocessing as the 10-feature version, applied to one date window."""
    w[RAW22_COLS] = w[RAW22_COLS].apply(pd.to_numeric)  # an all-NULL column arrives as object dtype
    # NULL recency = no outbreak within the 365-day lookback -> 400 ("longer ago than we can see")
    for c in ["days_since_outbreak_county", "days_since_outbreak_neighbor"]:
        w[c] = w[c].fillna(400.0)
    # NASS suppresses some county inventories -> fill with the state median (computed over the whole table)
    w["layer_inventory"] = w["layer_inventory"].fillna(w["state"].map(STATE_LAYER_MEDIAN))
    # log1p the right-skewed counts (clip guards against negatives)
    for c in LOG22_COLS:
        w[f"log_{c}"] = np.log1p(w[c].clip(lower=0))

    ids = pd.DataFrame({
        "fips": w["fips"].astype("int32"),                    # "09001" -> 9001
        "state": w["state"].map(STATE_CODES).astype("int16"),
        "days_from_present": (LAST_DAY - pd.to_datetime(w["day"])).dt.days.astype("float32"),
        TARGET: w[TARGET].astype("float32"),
    })
    return pd.concat([ids, w[TOP22_FEATURES].astype("float32")], axis=1)

In [10]:
TOP22_CACHE = "/content/drive/MyDrive/top22features_processed.parquet"
WINDOW_DAYS = 90  # rows per query ~ 3,100 counties x WINDOW_DAYS -- lower this if RAM is still tight

if os.path.exists(TOP22_CACHE):
    df22 = pd.read_parquet(TOP22_CACHE)
else:
    # needs `engine` from the connection cell at the top of the notebook.
    # One small read-only query for the stats that need the WHOLE table: the date range
    # (days_from_present counts back from the last day) and each state's median layer inventory.
    stats = pd.read_sql("""
        SELECT state,
               percentile_cont(0.5) WITHIN GROUP (ORDER BY layer_inventory) AS layer_median,
               min(day) AS first_day, max(day) AS last_day
        FROM feature_county_day
        WHERE is_conus AND target_complete_7d
        GROUP BY state
    """, engine)
    FIRST_DAY = pd.to_datetime(stats["first_day"]).min()
    LAST_DAY = pd.to_datetime(stats["last_day"]).max()
    STATE_LAYER_MEDIAN = stats.set_index("state")["layer_median"]
    STATE_CODES = {s: i for i, s in enumerate(sorted(stats["state"]))}

    WINDOW_QUERY = sqlalchemy.text(f"""
        SELECT fips, day, state, {TARGET}, {", ".join(RAW22_COLS)}
        FROM feature_county_day
        WHERE is_conus
          AND target_complete_7d
          AND day >= :start AND day < :end
    """)

    pieces = []
    start = FIRST_DAY
    while start <= LAST_DAY:
        end = start + pd.Timedelta(days=WINDOW_DAYS)
        w = pd.read_sql(WINDOW_QUERY, engine, params={"start": start.date(), "end": end.date()})
        pieces.append(preprocess_window(w))
        del w  # drop the raw window before fetching the next one
        print(f"{start.date()} .. {(end - pd.Timedelta(days=1)).date()}: {len(pieces[-1]):>8,} rows")
        start = end

    df22 = pd.concat(pieces, ignore_index=True)
    del pieces
    gc.collect()
    df22.to_parquet(TOP22_CACHE, index=False)

print(f"{len(df22):,} rows x {df22.shape[1]} columns | {df22.memory_usage().sum() / 1e9:.2f} GB in memory")
# remaining NULLs (expected: weather only) are filled after the split with train medians
print("share of rows still missing:")
print(df22[TOP22_FEATURES].isna().mean().loc[lambda s: s > 0].map("{:.2%}".format).to_string())

7,418,796 rows x 26 columns | 0.76 GB in memory
share of rows still missing:
temp_max_7d_c    0.08%
temp_min_7d_c    0.08%
precip_7d_mm     0.08%


In [11]:
# same split boundaries as the 10-feature model -- only the column list changes
KEEP22 = ["target_outbreak_next_7d", "fips", "state", "days_from_present"] + TOP22_FEATURES
df22_train, df22_val, df22_test = create_train_val_test(
    df22, train_percent=0.50, val_percent=0.25, week_total=340, columns=KEEP22
)

# fill any remaining NULLs with the TRAIN median, so val/test statistics never leak in
train_median = df22_train[TOP22_FEATURES].median()
for split in (df22_train, df22_val, df22_test):
    split[TOP22_FEATURES] = split[TOP22_FEATURES].fillna(train_median)
    # StandardScaler lets NaN through silently and a single NaN turns the loss into NaN -- check now
    assert split[TOP22_FEATURES].notna().all().all(), "NaN left in the 22-feature split"

del df22
gc.collect()

max days 2380
train start 2380 train stop 1190 validate start 1183 validate stop 588 test start 581 test stop 0.0
2380.0 1190.0
1183.0 588.0
581.0 0.0


110

In [12]:
# fit on train only -- val/test are transformed with the train mean/std
mlp22_scaler = StandardScaler().fit(df22_train[TOP22_FEATURES])

def to_tensors22(df_split):
    X = torch.tensor(mlp22_scaler.transform(df_split[TOP22_FEATURES]), dtype=torch.float32)
    y = torch.tensor((df_split[TARGET] > 0).to_numpy(), dtype=torch.float32).unsqueeze(1)  # (n, 1)
    return X, y

# train_mlp() reads the global X_train / X_val, so point them at the 22-feature tensors.
# To go back to the 10-feature model, re-run the tensor cell in the MLP section above.
X_train, y_train = to_tensors22(df22_train)
X_val, y_val = to_tensors22(df22_val)
X_test, y_test = to_tensors22(df22_test)

for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:5s} rows: {len(X):>9,} | features: {X.shape[1]} | positives: {int(y.sum()):>5,} | positive rate: {y.mean().item():.4%}")

train rows: 3,701,628 | features: 22 | positives: 4,817 | positive rate: 0.1301%
val   rows: 1,852,368 | features: 22 | positives: 2,645 | positive rate: 0.1428%
test  rows: 1,808,856 | features: 22 | positives: 4,858 | positive rate: 0.2686%


In [ ]:
# same settings as the 10-feature baseline `mlp_model` (best val PR-AUC 0.0649, lift 45.4x),
# so the comparison isolates the effect of the extra features
mlp_model_top22 = train_mlp(epochs=10)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0125 | Val PR-AUC: 0.0683 | Val base rate: 0.1428% | Val Lift: 47.8x | Val ROC-AUC: 0.8636
Epoch 2/10 | BCE Loss: 0.0073 | Val PR-AUC: 0.0668 | Val base rate: 0.1428% | Val Lift: 46.8x | Val ROC-AUC: 0.8707
Epoch 3/10 | BCE Loss: 0.0072 | Val PR-AUC: 0.0573 | Val base rate: 0.1428% | Val Lift: 40.2x | Val ROC-AUC: 0.8620
Epoch 4/10 | BCE Loss: 0.0071 | Val PR-AUC: 0.0597 | Val base rate: 0.1428% | Val Lift: 41.8x | Val ROC-AUC: 0.8719
Epoch 5/10 | BCE Loss: 0.0071 | Val PR-AUC: 0.0612 | Val base rate: 0.1428% | Val Lift: 42.9x | Val ROC-AUC: 0.8698
Epoch 6/10 | BCE Loss: 0.0070 | Val PR-AUC: 0.0630 | Val base rate: 0.1428% | Val Lift: 44.1x | Val ROC-AUC: 0.8704
Epoch 7/10 | BCE Loss: 0.0070 | Val PR-AUC: 0.0501 | Val base rate: 0.1428% | Val Lift: 35.1x | Val ROC-AUC: 0.8724
Epoch 8/10 | BCE Loss: 0.0070 | Val PR-AUC: 0.0553 | Val base rate: 0.1428% | Val Lift: 38.7x | Val ROC-AUC: 0.8555
Epoch 9/10 | BCE Loss: 0.0070 | Val PR-AUC: 0.0

In [ ]:
mlp_model_top22_dropout_0_5 = train_mlp(epochs=10, dropout=0.5)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0142 | Val PR-AUC: 0.0586 | Val base rate: 0.1428% | Val Lift: 41.0x | Val ROC-AUC: 0.8696
Epoch 2/10 | BCE Loss: 0.0082 | Val PR-AUC: 0.0543 | Val base rate: 0.1428% | Val Lift: 38.0x | Val ROC-AUC: 0.8793
Epoch 3/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0551 | Val base rate: 0.1428% | Val Lift: 38.6x | Val ROC-AUC: 0.8715
Epoch 4/10 | BCE Loss: 0.0076 | Val PR-AUC: 0.0502 | Val base rate: 0.1428% | Val Lift: 35.2x | Val ROC-AUC: 0.8749
Epoch 5/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0443 | Val base rate: 0.1428% | Val Lift: 31.0x | Val ROC-AUC: 0.8767
Epoch 6/10 | BCE Loss: 0.0074 | Val PR-AUC: 0.0681 | Val base rate: 0.1428% | Val Lift: 47.7x | Val ROC-AUC: 0.8735
Epoch 7/10 | BCE Loss: 0.0074 | Val PR-AUC: 0.0666 | Val base rate: 0.1428% | Val Lift: 46.6x | Val ROC-AUC: 0.8794
Epoch 8/10 | BCE Loss: 0.0073 | Val PR-AUC: 0.0689 | Val base rate: 0.1428% | Val Lift: 48.2x | Val ROC-AUC: 0.8688
Epoch 9/10 | BCE Loss: 0.0073 | Val PR-AUC: 0.0

In [ ]:
mlp_model_top22_dropout_0_5_lr_0_2 = train_mlp(epochs=10, dropout=0.5, lr=1e-2)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0096 | Val PR-AUC: 0.0265 | Val base rate: 0.1428% | Val Lift: 18.5x | Val ROC-AUC: 0.8627
Epoch 2/10 | BCE Loss: 0.0082 | Val PR-AUC: 0.0179 | Val base rate: 0.1428% | Val Lift: 12.6x | Val ROC-AUC: 0.8627
Epoch 3/10 | BCE Loss: 0.0081 | Val PR-AUC: 0.0570 | Val base rate: 0.1428% | Val Lift: 39.9x | Val ROC-AUC: 0.8730
Epoch 4/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0329 | Val base rate: 0.1428% | Val Lift: 23.1x | Val ROC-AUC: 0.8722
Epoch 5/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0600 | Val base rate: 0.1428% | Val Lift: 42.0x | Val ROC-AUC: 0.8717
Epoch 6/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0612 | Val base rate: 0.1428% | Val Lift: 42.9x | Val ROC-AUC: 0.8786
Epoch 7/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0182 | Val base rate: 0.1428% | Val Lift: 12.8x | Val ROC-AUC: 0.8831
Epoch 8/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0554 | Val base rate: 0.1428% | Val Lift: 38.8x | Val ROC-AUC: 0.8494
Epoch 9/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0

In [ ]:
mlp_model_top22_lr_0_2 = train_mlp(epochs=10, lr=1e-2)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0086 | Val PR-AUC: 0.0350 | Val base rate: 0.1428% | Val Lift: 24.5x | Val ROC-AUC: 0.8417
Epoch 2/10 | BCE Loss: 0.0077 | Val PR-AUC: 0.0575 | Val base rate: 0.1428% | Val Lift: 40.3x | Val ROC-AUC: 0.8556
Epoch 3/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0623 | Val base rate: 0.1428% | Val Lift: 43.7x | Val ROC-AUC: 0.8673
Epoch 4/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0590 | Val base rate: 0.1428% | Val Lift: 41.3x | Val ROC-AUC: 0.8766
Epoch 5/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0751 | Val base rate: 0.1428% | Val Lift: 52.6x | Val ROC-AUC: 0.8668
Epoch 6/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0753 | Val base rate: 0.1428% | Val Lift: 52.7x | Val ROC-AUC: 0.8769
Epoch 7/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0640 | Val base rate: 0.1428% | Val Lift: 44.9x | Val ROC-AUC: 0.8810
Epoch 8/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0687 | Val base rate: 0.1428% | Val Lift: 48.1x | Val ROC-AUC: 0.8278
Epoch 9/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0

In [ ]:
mlp_model_top22_hidden_128_64_32 = train_mlp(epochs=10, hidden=(128, 64, 32))

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0114 | Val PR-AUC: 0.0684 | Val base rate: 0.1428% | Val Lift: 47.9x | Val ROC-AUC: 0.8761
Epoch 2/10 | BCE Loss: 0.0073 | Val PR-AUC: 0.0706 | Val base rate: 0.1428% | Val Lift: 49.5x | Val ROC-AUC: 0.8763
Epoch 3/10 | BCE Loss: 0.0072 | Val PR-AUC: 0.0681 | Val base rate: 0.1428% | Val Lift: 47.7x | Val ROC-AUC: 0.8736
Epoch 4/10 | BCE Loss: 0.0071 | Val PR-AUC: 0.0650 | Val base rate: 0.1428% | Val Lift: 45.5x | Val ROC-AUC: 0.8695
Epoch 5/10 | BCE Loss: 0.0070 | Val PR-AUC: 0.0630 | Val base rate: 0.1428% | Val Lift: 44.1x | Val ROC-AUC: 0.8700
Epoch 6/10 | BCE Loss: 0.0069 | Val PR-AUC: 0.0671 | Val base rate: 0.1428% | Val Lift: 47.0x | Val ROC-AUC: 0.8759
Epoch 7/10 | BCE Loss: 0.0069 | Val PR-AUC: 0.0599 | Val base rate: 0.1428% | Val Lift: 41.9x | Val ROC-AUC: 0.8606
Epoch 8/10 | BCE Loss: 0.0068 | Val PR-AUC: 0.0648 | Val base rate: 0.1428% | Val Lift: 45.4x | Val ROC-AUC: 0.8729
Epoch 9/10 | BCE Loss: 0.0068 | Val PR-AUC: 0.0

In [ ]:
mlp_model_top22_hidden_128_64_32_lr_1_2 = train_mlp(epochs=10, hidden=(128, 64, 32), lr=1e-2)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0090 | Val PR-AUC: 0.0441 | Val base rate: 0.1428% | Val Lift: 30.9x | Val ROC-AUC: 0.8810
Epoch 2/10 | BCE Loss: 0.0079 | Val PR-AUC: 0.0729 | Val base rate: 0.1428% | Val Lift: 51.1x | Val ROC-AUC: 0.8856
Epoch 3/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0727 | Val base rate: 0.1428% | Val Lift: 50.9x | Val ROC-AUC: 0.8630
Epoch 4/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0761 | Val base rate: 0.1428% | Val Lift: 53.3x | Val ROC-AUC: 0.8794
Epoch 5/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0715 | Val base rate: 0.1428% | Val Lift: 50.1x | Val ROC-AUC: 0.8643
Epoch 6/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0688 | Val base rate: 0.1428% | Val Lift: 48.2x | Val ROC-AUC: 0.8671
Epoch 7/10 | BCE Loss: 0.0077 | Val PR-AUC: 0.0759 | Val base rate: 0.1428% | Val Lift: 53.2x | Val ROC-AUC: 0.8550
Epoch 8/10 | BCE Loss: 0.0079 | Val PR-AUC: 0.0761 | Val base rate: 0.1428% | Val Lift: 53.3x | Val ROC-AUC: 0.8744
Epoch 9/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0

In [ ]:
results = []
for hidden in [(64, 32), (128, 64), (256, 128)]:
    for seed in [0, 1]:
        m = train_mlp(hidden=hidden, seed=seed, lr=1e-2)
        ap, base, lift = pr_auc_lift(y_val, predict_logits(m, X_val))
        results.append({"hidden": str(hidden), "seed": seed, "val_pr_auc": ap, "lift": lift})

pd.DataFrame(results).groupby("hidden")[["val_pr_auc", "lift"]].agg(["mean", "std"])

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0086 | Val PR-AUC: 0.0350 | Val base rate: 0.1428% | Val Lift: 24.5x | Val ROC-AUC: 0.8417
Epoch 2/10 | BCE Loss: 0.0077 | Val PR-AUC: 0.0575 | Val base rate: 0.1428% | Val Lift: 40.3x | Val ROC-AUC: 0.8556
Epoch 3/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0623 | Val base rate: 0.1428% | Val Lift: 43.7x | Val ROC-AUC: 0.8673
Epoch 4/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0590 | Val base rate: 0.1428% | Val Lift: 41.3x | Val ROC-AUC: 0.8766
Epoch 5/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0751 | Val base rate: 0.1428% | Val Lift: 52.6x | Val ROC-AUC: 0.8668
Epoch 6/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0753 | Val base rate: 0.1428% | Val Lift: 52.7x | Val ROC-AUC: 0.8769
Epoch 7/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0640 | Val base rate: 0.1428% | Val Lift: 44.9x | Val ROC-AUC: 0.8810
Epoch 8/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0687 | Val base rate: 0.1428% | Val Lift: 48.1x | Val ROC-AUC: 0.8278
Epoch 9/10 | BCE Loss: 0.0075 | Val PR-AUC: 0.0

val_pr_auc                 lift          
                 mean       std       mean       std
hidden                                              
(128, 64)    0.076360  0.000770  53.477159  0.539559
(256, 128)   0.077244  0.000142  54.096017  0.099098
(64, 32)     0.074744  0.000783  52.345505  0.548181

In [13]:
mlp_model_top22_hidden_128_64_32_lr_1_2 = train_mlp(epochs=10, hidden=(256, 128, 64, 32), lr=1e-2)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0090 | Val PR-AUC: 0.0387 | Val base rate: 0.1428% | Val Lift: 27.1x | Val ROC-AUC: 0.8698
Epoch 2/10 | BCE Loss: 0.0083 | Val PR-AUC: 0.0095 | Val base rate: 0.1428% | Val Lift: 6.7x | Val ROC-AUC: 0.8302
Epoch 3/10 | BCE Loss: 0.0085 | Val PR-AUC: 0.0075 | Val base rate: 0.1428% | Val Lift: 5.2x | Val ROC-AUC: 0.8589
Epoch 4/10 | BCE Loss: 0.0086 | Val PR-AUC: 0.0096 | Val base rate: 0.1428% | Val Lift: 6.7x | Val ROC-AUC: 0.8691
Epoch 5/10 | BCE Loss: 0.0085 | Val PR-AUC: 0.0081 | Val base rate: 0.1428% | Val Lift: 5.7x | Val ROC-AUC: 0.8484
Epoch 6/10 | BCE Loss: 0.0087 | Val PR-AUC: 0.0056 | Val base rate: 0.1428% | Val Lift: 3.9x | Val ROC-AUC: 0.8411
Epoch 7/10 | BCE Loss: 0.0087 | Val PR-AUC: 0.0066 | Val base rate: 0.1428% | Val Lift: 4.6x | Val ROC-AUC: 0.8580
Epoch 8/10 | BCE Loss: 0.0085 | Val PR-AUC: 0.0085 | Val base rate: 0.1428% | Val Lift: 5.9x | Val ROC-AUC: 0.8463


KeyboardInterrupt: 

In [15]:
model_10 = train_mlp(epochs=10, hidden=(256, 128, 64, 32))

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0098 | Val PR-AUC: 0.0653 | Val base rate: 0.1428% | Val Lift: 45.7x | Val ROC-AUC: 0.8724
Epoch 2/10 | BCE Loss: 0.0073 | Val PR-AUC: 0.0670 | Val base rate: 0.1428% | Val Lift: 46.9x | Val ROC-AUC: 0.8615
Epoch 3/10 | BCE Loss: 0.0071 | Val PR-AUC: 0.0700 | Val base rate: 0.1428% | Val Lift: 49.0x | Val ROC-AUC: 0.8786
Epoch 4/10 | BCE Loss: 0.0070 | Val PR-AUC: 0.0634 | Val base rate: 0.1428% | Val Lift: 44.4x | Val ROC-AUC: 0.8686
Epoch 5/10 | BCE Loss: 0.0069 | Val PR-AUC: 0.0651 | Val base rate: 0.1428% | Val Lift: 45.6x | Val ROC-AUC: 0.8680
Epoch 6/10 | BCE Loss: 0.0069 | Val PR-AUC: 0.0700 | Val base rate: 0.1428% | Val Lift: 49.0x | Val ROC-AUC: 0.8707
Epoch 7/10 | BCE Loss: 0.0069 | Val PR-AUC: 0.0580 | Val base rate: 0.1428% | Val Lift: 40.6x | Val ROC-AUC: 0.8714
Epoch 8/10 | BCE Loss: 0.0068 | Val PR-AUC: 0.0680 | Val base rate: 0.1428% | Val Lift: 47.7x | Val ROC-AUC: 0.8565
Epoch 9/10 | BCE Loss: 0.0067 | Val PR-AUC: 0.0

In [14]:
mlp_model_top22_hidden_128_64_32_lr_1_2 = train_mlp(epochs=10, hidden=(256, 128, 64), lr=1e-2)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0088 | Val PR-AUC: 0.0372 | Val base rate: 0.1428% | Val Lift: 26.1x | Val ROC-AUC: 0.8646
Epoch 2/10 | BCE Loss: 0.0081 | Val PR-AUC: 0.0547 | Val base rate: 0.1428% | Val Lift: 38.3x | Val ROC-AUC: 0.8596
Epoch 3/10 | BCE Loss: 0.0082 | Val PR-AUC: 0.0075 | Val base rate: 0.1428% | Val Lift: 5.2x | Val ROC-AUC: 0.8459
Epoch 4/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0107 | Val base rate: 0.1428% | Val Lift: 7.5x | Val ROC-AUC: 0.8474
Epoch 5/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0771 | Val base rate: 0.1428% | Val Lift: 54.0x | Val ROC-AUC: 0.8770
Epoch 6/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0729 | Val base rate: 0.1428% | Val Lift: 51.0x | Val ROC-AUC: 0.8811
Epoch 7/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0775 | Val base rate: 0.1428% | Val Lift: 54.3x | Val ROC-AUC: 0.8763
Epoch 8/10 | BCE Loss: 0.0081 | Val PR-AUC: 0.0719 | Val base rate: 0.1428% | Val Lift: 50.4x | Val ROC-AUC: 0.8760
Epoch 9/10 | BCE Loss: 0.0079 | Val PR-AUC: 0.074

In [16]:
model_11 = train_mlp(epochs=10, hidden=(512, 256, 128), lr=1e-2)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0090 | Val PR-AUC: 0.0287 | Val base rate: 0.1428% | Val Lift: 20.1x | Val ROC-AUC: 0.8746
Epoch 2/10 | BCE Loss: 0.0082 | Val PR-AUC: 0.0627 | Val base rate: 0.1428% | Val Lift: 43.9x | Val ROC-AUC: 0.8514
Epoch 3/10 | BCE Loss: 0.0092 | Val PR-AUC: 0.0766 | Val base rate: 0.1428% | Val Lift: 53.7x | Val ROC-AUC: 0.8495
Epoch 4/10 | BCE Loss: 0.0088 | Val PR-AUC: 0.0777 | Val base rate: 0.1428% | Val Lift: 54.4x | Val ROC-AUC: 0.8741
Epoch 5/10 | BCE Loss: 0.0090 | Val PR-AUC: 0.0074 | Val base rate: 0.1428% | Val Lift: 5.2x | Val ROC-AUC: 0.8604
Epoch 6/10 | BCE Loss: 0.0086 | Val PR-AUC: 0.0067 | Val base rate: 0.1428% | Val Lift: 4.7x | Val ROC-AUC: 0.8489
Epoch 7/10 | BCE Loss: 0.0089 | Val PR-AUC: 0.0783 | Val base rate: 0.1428% | Val Lift: 54.8x | Val ROC-AUC: 0.8757
Epoch 8/10 | BCE Loss: 0.0088 | Val PR-AUC: 0.0124 | Val base rate: 0.1428% | Val Lift: 8.7x | Val ROC-AUC: 0.8299
Epoch 9/10 | BCE Loss: 0.0092 | Val PR-AUC: 0.0051

In [17]:
model_12 = train_mlp(epochs=10, hidden=(512, 256), lr=1e-2)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0088 | Val PR-AUC: 0.0385 | Val base rate: 0.1428% | Val Lift: 27.0x | Val ROC-AUC: 0.8150
Epoch 2/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0581 | Val base rate: 0.1428% | Val Lift: 40.7x | Val ROC-AUC: 0.8739
Epoch 3/10 | BCE Loss: 0.0082 | Val PR-AUC: 0.0656 | Val base rate: 0.1428% | Val Lift: 45.9x | Val ROC-AUC: 0.8429
Epoch 4/10 | BCE Loss: 0.0081 | Val PR-AUC: 0.0790 | Val base rate: 0.1428% | Val Lift: 55.3x | Val ROC-AUC: 0.8701
Epoch 5/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0724 | Val base rate: 0.1428% | Val Lift: 50.7x | Val ROC-AUC: 0.8701
Epoch 6/10 | BCE Loss: 0.0081 | Val PR-AUC: 0.0762 | Val base rate: 0.1428% | Val Lift: 53.4x | Val ROC-AUC: 0.8746
Epoch 7/10 | BCE Loss: 0.0080 | Val PR-AUC: 0.0504 | Val base rate: 0.1428% | Val Lift: 35.3x | Val ROC-AUC: 0.8307
Epoch 8/10 | BCE Loss: 0.0081 | Val PR-AUC: 0.0072 | Val base rate: 0.1428% | Val Lift: 5.1x | Val ROC-AUC: 0.8289
Epoch 9/10 | BCE Loss: 0.0081 | Val PR-AUC: 0.04

In [22]:
model_13 = train_mlp(epochs=10, hidden=(256, 128), lr=1e-2)

Training on device: cuda
Epoch 1/10 | BCE Loss: 0.0085 | Val PR-AUC: 0.0619 | Val base rate: 0.1428% | Val Lift: 43.3x | Val ROC-AUC: 0.8786
Epoch 2/10 | BCE Loss: 0.0077 | Val PR-AUC: 0.0697 | Val base rate: 0.1428% | Val Lift: 48.8x | Val ROC-AUC: 0.8621
Epoch 3/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0699 | Val base rate: 0.1428% | Val Lift: 49.0x | Val ROC-AUC: 0.8733
Epoch 4/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0541 | Val base rate: 0.1428% | Val Lift: 37.9x | Val ROC-AUC: 0.8705
Epoch 5/10 | BCE Loss: 0.0077 | Val PR-AUC: 0.0732 | Val base rate: 0.1428% | Val Lift: 51.3x | Val ROC-AUC: 0.8488
Epoch 6/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0053 | Val base rate: 0.1428% | Val Lift: 3.7x | Val ROC-AUC: 0.8325
Epoch 7/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0735 | Val base rate: 0.1428% | Val Lift: 51.5x | Val ROC-AUC: 0.8785
Epoch 8/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.0736 | Val base rate: 0.1428% | Val Lift: 51.6x | Val ROC-AUC: 0.8735
Epoch 9/10 | BCE Loss: 0.0078 | Val PR-AUC: 0.07

In [18]:
def test_(model):
    test_pr_auc, test_base, test_lift = pr_auc_lift(y_test, predict_logits(model, X_test))
    print(f"MLP test PR-AUC: {test_pr_auc:.4f} | base rate: {test_base:.4%} | lift: {test_lift:.1f}x")

In [20]:
test_(model_12)
test_(model_11)
test_(model_10)

MLP test PR-AUC: 0.1286 | base rate: 0.2686% | lift: 47.9x
MLP test PR-AUC: 0.1309 | base rate: 0.2686% | lift: 48.7x
MLP test PR-AUC: 0.1300 | base rate: 0.2686% | lift: 48.4x


In [23]:
test_(mlp_model_top22_hidden_128_64_32_lr_1_2)
test_(model_13)

MLP test PR-AUC: 0.1270 | base rate: 0.2686% | lift: 47.3x
MLP test PR-AUC: 0.1340 | base rate: 0.2686% | lift: 49.9x


## Next step is to try Transformer based model with similar dataset as Simple Neural Network